# Imports

In [ ]:
import json
import os
from copy import deepcopy

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM, LlamaModel
from tqdm import tqdm

# Load tokenizers

In [ ]:
OLD_TOKENIZER_PATH = "/home/jovyan/tokenizers/llama_tokenizers/extended_cc_10k/"
NEW_TOKENIZER_PATH = "/home/jovyan/tokenizers/llama_tokenizers/llama_tokenizer_extended_cc_10k_ar_10k_uz_5k_kk_7k"

# NEW_TOKENIZER_PATH = "/home/jovyan/saverkiev/6_tokenizers/tokenizers/tokenizers_new/hf_128000_gs3.5_latex_spaces_v12_gpt4o_rx_code_added_tokens"

In [ ]:
tokenizer_old = AutoTokenizer.from_pretrained(OLD_TOKENIZER_PATH)
tokenizer_new = AutoTokenizer.from_pretrained(NEW_TOKENIZER_PATH)

In [ ]:
tokenizer_old

In [ ]:
tokenizer_new

## Explore vocabs

In [ ]:
# tokenizer_old.vocab

In [ ]:
# tokenizer_new.vocab

In [ ]:
len(tokenizer_old)

In [ ]:
len(tokenizer_new)

## Get tokens in text representation

### Get intersecting tokens

In [ ]:
tokenizer_old_tokens_set = set(tokenizer_old.vocab)
tokenizer_new_tokens_set = set(tokenizer_new.vocab)

In [ ]:
len(tokenizer_new_tokens_set.intersection(tokenizer_old_tokens_set))

In [ ]:
tokenizer_new_added_tokens = list(tokenizer_new_tokens_set - tokenizer_new_tokens_set.intersection(tokenizer_old_tokens_set))

In [ ]:
tokenizer_new_added_tokens[:10]

In [ ]:
tokenizer_new_added_tokens_by_id = [
    (idx, token) for token, idx in tokenizer_new.vocab.items()
    if token in tokenizer_new_added_tokens
]

In [ ]:
tokenizer_new_added_tokens_by_id[:15]

In [ ]:
tokenizer_new_added_tokens_by_id = sorted(tokenizer_new_added_tokens_by_id, key=lambda x: x[0])

In [ ]:
tokenizer_new_added_tokens_by_id[:10]

---

In [ ]:
tokenizer_new_text_tokens = [
    tokenizer_new.decode([token_idx], clean_up_tokenization_spaces=False, skip_special_tokens=False)
    for token_idx in range(len(tokenizer_new.vocab))
]

In [ ]:
tokenizer_new_text_tokens_decoder = [
    tokenizer_new.decoder.decode([token_str])
    for token_str, token_id in sorted(
        tokenizer_new.vocab.items(), key=lambda item: item[1]
    )
]

In [ ]:
tokenizer_new_text_tokens == tokenizer_new_text_tokens_decoder

In [ ]:
all(tokenizer_new_text_tokens[i] == tokenizer_new_text_tokens_decoder[i] for i in range(len(tokenizer_new_text_tokens)))

In [ ]:
tokenizer_old_text_tokens = [
    tokenizer_old.decode([token_idx], clean_up_tokenization_spaces=False, skip_special_tokens=False)
    for token_idx in range(len(tokenizer_old.vocab))
]

In [ ]:
# tokenizer_old_text_tokens

In [ ]:
# tokenizer_new_text_tokens

In [ ]:
intersecting_tokens = set(tokenizer_new_text_tokens).intersection(set(tokenizer_old_text_tokens))

In [ ]:
# intersecting_tokens

In [ ]:
len(intersecting_tokens)

# Load model checkpoint

In [ ]:
# CKPT_PATH = "/home/jovyan/ndermolaev/tokenizers/checkpoint_extended/ep0-ba180000-rank0.pt"

In [ ]:
CKPT_PATH = "/home/jovyan/malkolesov/checkpoints/gigaseek-20b-8k-437_5k_step-packed.pt"

In [ ]:
CKPT_PATH = "/home/jovyan/vmamedov/checkpoints_prod/gigaseek_20b_8k/run_12april/ep1-ba399000-rank0.pt"

In [ ]:
ckpt = torch.load(CKPT_PATH)

In [ ]:
# ckpt

In [ ]:
model_state_dict = ckpt["state"]["model"]

In [ ]:
emb_weight = model_state_dict["model.model.embed_tokens.weight"]

In [ ]:
lm_head_weight = model_state_dict["model.lm_head.weight"]

In [ ]:
emb_weight.shape, lm_head_weight.shape

In [ ]:
hidden_size = emb_weight.shape[1]

In [ ]:
emb_weight.dtype

In [ ]:
ckpt.keys()

# Get embeddings for tokens

In [ ]:
def get_token_embedding(token: str, embedding_matrix, tokenizer) -> torch.Tensor:
    # add `\n` to the beginning of the token to get rid of the whitespace in the begining
    # tokenized result would be [added whitespace token, newline token, tokenized_tokens]
    token_tokenized = tokenizer(
        '\n' + token,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"][0][2:]
    token_embedding = embedding_matrix[token_tokenized].mean(axis=0)
    
    return token_embedding

## Embedding layer

In [ ]:
new_emb_weight = torch.empty((len(tokenizer_new_text_tokens), hidden_size), dtype=emb_weight.dtype)
torch.nn.init.xavier_normal_(new_emb_weight)

In [ ]:
# new_emb_weight = torch.zeros((len(tokenizer_new_text_tokens), hidden_size), dtype=emb_weight.dtype)

In [ ]:
for idx, token in tqdm(enumerate(tokenizer_new_text_tokens), total=len(tokenizer_new_text_tokens)):
    new_emb_weight[idx] = get_token_embedding(token, emb_weight, tokenizer_old)

In [ ]:
# Zero out added reserved special tokens
# new_emb_weight[-256:] = 0

In [ ]:
new_emb_weight[-256:]

---

### Init added tokens only

In [ ]:
new_emb_weight = torch.zeros((len(tokenizer_new.vocab), hidden_size), dtype=emb_weight.dtype)
for token_idx in tqdm(tokenizer_old.vocab.values()):
    new_emb_weight[token_idx] = emb_weight[token_idx]

for token_idx, token in tqdm(tokenizer_new_added_tokens_by_id):
    new_emb_weight[token_idx] = get_token_embedding(token, emb_weight, tokenizer_old)

## LM head layer

In [ ]:
new_lm_head = torch.empty((len(tokenizer_new_text_tokens), hidden_size), dtype=emb_weight.dtype)
torch.nn.init.xavier_normal_(new_lm_head)

In [ ]:
# new_lm_head = torch.zeros((len(tokenizer_new_text_tokens), hidden_size), dtype=lm_head_weight.dtype)

In [ ]:
for idx, token in tqdm(enumerate(tokenizer_new_text_tokens), total=len(tokenizer_new_text_tokens)):
    new_lm_head[idx] = get_token_embedding(token, lm_head_weight, tokenizer_old)

In [ ]:
# Zero out added reserved special tokens
# new_lm_head[-256:] = 0

In [ ]:
new_lm_head[-256:]

### Init added tokens only

In [ ]:
new_lm_head = torch.zeros((len(tokenizer_new.vocab), hidden_size), dtype=lm_head_weight.dtype)
for token_idx in tqdm(tokenizer_old.vocab.values()):
    new_lm_head[token_idx] = lm_head_weight[token_idx]

for token_idx, token in tqdm(tokenizer_new_added_tokens_by_id):
    new_lm_head[token_idx] = get_token_embedding(token, lm_head_weight, tokenizer_old)

# Update model checkpoint

In [ ]:
ckpt["state"].keys()

In [ ]:
ckpt["state"]["model"]["model.model.embed_tokens.weight"].shape

In [ ]:
ckpt["state"]["model"]["model.model.embed_tokens.weight"] = new_emb_weight

In [ ]:
ckpt["state"]["model"]["model.model.embed_tokens.weight"].shape

In [ ]:
ckpt["state"]["model"]["model.lm_head.weight"].shape

In [ ]:
ckpt["state"]["model"]["model.lm_head.weight"] = new_lm_head

In [ ]:
ckpt["state"]["model"]["model.lm_head.weight"].shape

## Update model and tokenizer config

### Model

In [ ]:
ckpt["state"].keys()

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["model"]["config"]

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["model"]["config"]["content"]["vocab_size"] = len(tokenizer_new.vocab)

### Tokenizer

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].keys()

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"]["special_tokens_map.json"]

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"]["tokenizer_config.json"]

In [ ]:
# Same as State does it during training
# for filename in ["special_tokens_map.json", "tokenizer.json", "tokenizer_config.json"]:
for filename in ["special_tokens_map.json", "tokenizer_config.json"]:
    with open(os.path.join(NEW_TOKENIZER_PATH, filename), "rb") as fin:
        content = json.load(fin)

    ckpt["state"]["integrations"]["huggingface"]["tokenizer"][filename.split(".")[0]] = {
        "file_extension": "." + filename.split(".")[1],
        "content": content
    }
    # ckpt["state"]["integrations"]["huggingface"]["tokenizer"][filename.split(".")[0]]["file_extension"] = ".json"
    # ckpt["state"]["integrations"]["huggingface"]["tokenizer"][filename.split(".")[0]]["content"] = content

for filename in ["tokenizer.model"]:
    import sentencepiece as sp
    sp_tok = sp.SentencePieceProcessor(model_file=os.path.join(NEW_TOKENIZER_PATH, filename))
    content = sp_tok.serialized_model_proto()

    ckpt["state"]["integrations"]["huggingface"]["tokenizer"][filename.split(".")[0]] = {
        "file_extension": "." + filename.split(".")[1],
        "content": content
    }

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"]["tokenizer_config"]

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"]["tokenizer"]

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"]["special_tokens_map"]

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].keys()

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].pop("special_tokens_map.json")
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].pop("tokenizer_config.json")
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].pop("tokenizer.model")

In [ ]:
ckpt["state"]["integrations"]["huggingface"]["tokenizer"].keys()

## Update optimizer state

In [ ]:
ckpt["state"].keys()

In [ ]:
ckpt["state"]["optimizers"].keys()

In [ ]:
ckpt["state"]["optimizers"]["DecoupledAdamW"].keys()

In [ ]:
ckpt["state"]["optimizers"]["DecoupledAdamW"]["param_groups"][0]

In [ ]:
ckpt["state"]["optimizers"]["DecoupledAdamW"]["state"]["model.model.embed_tokens.weight"]

In [ ]:
ckpt["state"]["optimizers"]["DecoupledAdamW"]["state"]["model.model.embed_tokens.weight"]["exp_avg"].shape

In [ ]:
layer_names = ['model.model.embed_tokens.weight', 'model.lm_head.weight']
for name in layer_names:
    for moment in ['exp_avg', 'exp_avg_sq']:
        new_moments = torch.zeros((len(tokenizer_new.vocab), hidden_size), dtype=emb_weight.dtype)
        # Zero moments for added tokens, same ones for existing ones
        new_moments[:len(tokenizer_old.vocab)] = ckpt["state"]["optimizers"]["DecoupledAdamW"]["state"][name][moment]
        ckpt["state"]["optimizers"]["DecoupledAdamW"]["state"][name][moment] = new_moments

In [ ]:
ckpt["state"]["optimizers"]["DecoupledAdamW"]["state"]["model.model.embed_tokens.weight"]

# Save checkpoint

In [ ]:
SAVE_PATH = "/home/jovyan/malkolesov/checkpoints/gigaseek-20b-8k-437_5k_step-packed-extended_61k_tokenizer_remapped.pt"

In [ ]:
SAVE_PATH = "/home/jovyan/malkolesov/checkpoints/gigaseek-20b-8k-399k_step-packed-extended_61k_tokenizer_remapped.pt"

In [ ]:
torch.save(ckpt, SAVE_PATH)